### Read Bronze data

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.customers"
SILVER_TABLE = "olist_mlops.silver.customers_clean"

bronze_customers = spark.table(BRONZE_TABLE)

print("Bronze rows:", bronze_customers.count())

bronze_customers.printSchema()

### Clean and standardize

In [0]:
silver_customers = (
    bronze_customers

    # Remove unnecessary spaces
    .withColumn(
        "customer_city",
        F.trim(F.col("customer_city"))
    )

    # Standardize state abbreviations
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )

    # Create standardized five-digit ZIP prefix
    .withColumn(
        "customer_zip_prefix_5d",
        F.lpad(
            F.trim(F.col("customer_zip_code_prefix")),
            5,
            "0"
        )
    )

    # Track Silver processing time
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

silver_customers.printSchema()

### Validate data quality

In [0]:
# 1. Row-count reconciliation

bronze_count = bronze_customers.count()
silver_count = silver_customers.count()

assert bronze_count == silver_count == 99441


# 2. Validate customer_id

null_ids = silver_customers.filter(
    F.col("customer_id").isNull()
).count()

duplicate_ids = (
    silver_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert null_ids == 0
assert duplicate_ids == 0


# 3. Validate ZIP-code prefixes

invalid_zip = silver_customers.filter(
    F.col("customer_zip_code_prefix").isNull()
    | ~F.trim(
        F.col("customer_zip_code_prefix")
    ).rlike(r"^[0-9]{1,5}$")
).count()

print("Invalid ZIP prefixes:", invalid_zip)

assert invalid_zip == 0


# 4. Print final validation

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Null customer IDs:", null_ids)
print("Duplicate customer IDs:", duplicate_ids)
print("Invalid ZIP prefixes:", invalid_zip)

print("CUSTOMERS VALIDATION PASSED")

### Write the Silver table

In [0]:
(
    silver_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

assert spark.table(SILVER_TABLE).count() == 99441

print("Created:", SILVER_TABLE)

### Explore the result using SQL

In [0]:
%sql

SELECT
    COUNT(*) AS total_customer_records,
    COUNT(DISTINCT customer_id) AS distinct_customer_ids,
    COUNT(DISTINCT customer_unique_id) AS unique_customers,
    COUNT(DISTINCT customer_state) AS total_states
FROM olist_mlops.silver.customers_clean;

In [0]:
%sql
-- inspect our ZIP normalization
SELECT
    customer_zip_code_prefix,
    customer_zip_prefix_5d,
    customer_city,
    customer_state
FROM olist_mlops.silver.customers_clean
WHERE LENGTH(customer_zip_code_prefix) < 5
LIMIT 10;